In [1]:
import pandas as pd
import seaborn as sns
df = sns.load_dataset("tips")
toplam_ciro = df['total_bill'].sum()
ortalama_hesap = df['total_bill'].mean()
df['bahsis_orani'] = df['tip'] / df['total_bill'] * 100
ortalama_bahsis_orani = df['bahsis_orani'].mean()
toplam_musteri = df['size'].sum()

print(f"Toplam Ciro: {toplam_ciro:.2f} TL")
print(f"Ortalama Hesap: {ortalama_hesap:.2f} TL")
print(f"Ortalama Bahşiş Oranı: %{ortalama_bahsis_orani:.1f}")
print(f"Toplam Müşteri: {toplam_musteri}")

Toplam Ciro: 4827.77 TL
Ortalama Hesap: 19.79 TL
Ortalama Bahşiş Oranı: %16.1
Toplam Müşteri: 627


In [3]:
import plotly.express as px
gunluk_ciro = df.groupby('day')['total_bill'].sum().reset_index()
fig1 = px.bar(gunluk_ciro, x='day', y='total_bill', title='Gün Bazında Toplam Ciro')
fig1.update_layout(
    xaxis_title = "Günler",
    yaxis_title = "Toplam Ücret"
)
fig1.show()

C:\Users\asus\AppData\Local\Temp\ipykernel_11988\1120793601.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [12]:
print(gunluk_ciro[(gunluk_ciro["day"]=="Fri") | (gunluk_ciro["day"]=="Thur")]["total_bill"].sum())
print(gunluk_ciro[(gunluk_ciro["day"]=="Sat") | (gunluk_ciro["day"]=="Sun")]["total_bill"].sum())

1422.21
3405.5600000000004


Gün bazında toplam ödenen ücretlere göz attığımda cumartesi ve pazar'ın perşembe ve cuma'ya kıyasla yaklaşık 2.5 kat daha fazla toplam ücret kazandırdığını gözlemliyoruz.

In [14]:
zaman_ciro = df.groupby('time', observed=True)['total_bill'].sum().reset_index()
fig2 = px.bar(zaman_ciro, x='time', y='total_bill', title='Öğle vs Akşam Ciro Karşılaştırması')
fig2.show()

In [ ]:
# Burada akşam cirosunun öğle cirosuna olan oranını tespit ediyoruz.
print(zaman_ciro[zaman_ciro['time']=="Dinner"]["total_bill"].sum() / zaman_ciro[zaman_ciro['time']=="Lunch"]["total_bill"].sum())

3.135241162513812


Restoranımız, akşamları öğlen vaktine kıyasla yaklaşık 3 kat daha fazla ciro yapmaktadır. Bu durumda akşam vakitleri ve öğlen vakitleri çalışan sayısını optimize etme, restoranı daha geç vakitte açma gibi seçenekler düşünülebilir.

In [22]:
fig3 = px.scatter(df, x='total_bill', y='tip', color='sex', title='Hesap Tutarı vs Bahşiş (Cinsiyete Göre)')
fig3.show()

In [23]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 2x2'lik bir grid oluşturuyoruz: üstte 2 bar chart, altta scatter (geniş)
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Gün Bazında Ciro", "Öğle vs Akşam Ciro", "Hesap Tutarı vs Bahşiş (Cinsiyete Göre)"),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "scatter", "colspan": 2}, None]]
)

# 1. Grafik: Gün bazında ciro (satır 1, sütun 1)
fig.add_trace(
    go.Bar(x=gunluk_ciro['day'], y=gunluk_ciro['total_bill'], name='Gün Bazında Ciro'),
    row=1, col=1
)

# 2. Grafik: Öğle vs Akşam (satır 1, sütun 2)
fig.add_trace(
    go.Bar(x=zaman_ciro['time'], y=zaman_ciro['total_bill'], name='Öğle vs Akşam'),
    row=1, col=2
)

# 3. Grafik: Hesap vs Bahşiş, cinsiyete göre (satır 2, geniş)
for cinsiyet in df['sex'].unique():
    alt_df = df[df['sex'] == cinsiyet]
    fig.add_trace(
        go.Scatter(x=alt_df['total_bill'], y=alt_df['tip'], mode='markers', name=cinsiyet),
        row=2, col=1
    )

fig.update_layout(
    title_text=f"Restoran Performans Paneli — Toplam Ciro: {toplam_ciro:.0f} TL | Ort. Bahşiş: %{ortalama_bahsis_orani:.1f}",
    height=700,
    showlegend=True
)

fig.show()